## 6. Generate the trees.

In [ ]:
!~/miniconda3/bin/conda run -n nextstrain augur filter \
    --metadata 

The comand block below was taken from the zika tutoral: 
https://github.com/nextstrain/zika-tutorial/blob/main/Snakefile

It runs 11 augur comands that will:
- curate metadata
- align the sequences
- infer a ML three
- perform ancestral recosntruction on the tree
- calculate the traits on the tree
- trasnlate the alignment into protein
- infer the frequencies for the clades/traits over the time
- export the auspice json to be visualized.

some changes must be made on it before running such as: 
- **path for the files**
- **coluns and branch-labels** at augur traits
- change the **REFSEQ** to the richt one for you virus.
- The lineages is not mandatory, unless you have a TSV files describing which mutation describes a lineage.
- change the **auspice_config.json** to match your needs.

### 229E

In [14]:
#!~/miniconda3/bin/conda env list

from pathlib import Path

# Variables
WORKDIR = Path("/Users/dt14/Desktop/HCoV/REFs/229E")

SEQ = WORKDIR / "NCBI_virus/229E_sequences_09Sep26_formated.fasta"
META = WORKDIR / "229E_metadata_final.txt"
REFSEQ = WORKDIR / "229E_NC_002645.gb"
COLORS = WORKDIR / "colors.tsv"
CONFIG = WORKDIR / "NCBI_virus/229E_auspice_config.json"
RESULTS = WORKDIR / "augur_results"
THREADS = 4
MIN_LEN = 21803

# Create output directory
RESULTS.mkdir(parents=True, exist_ok=True)

# 1. Running augur index
print("1 - Running augur index")

!~/miniconda3/bin/conda run -n nextstrain augur index \
    --sequences "{SEQ}" \
    --output "{RESULTS}/sequence_index.tsv"

# 2. Running augur filter
print("2 - Running augur filter")

!~/miniconda3/bin/conda run -n nextstrain augur filter \
    --sequences "{SEQ}" \
    --sequence-index "{RESULTS}/sequence_index.tsv" \
    --min-length  "{MIN_LEN}" \
    --metadata "{META}" \
    --output-sequences "{RESULTS}/filtered_sequences.fasta" \
    --output-metadata "{RESULTS}/filtered_metadata.tsv"

1 - Running augur index
2 - Running augur filter
12 strains were dropped during filtering
	12 were dropped because they were shorter than the minimum length of 21803bp when only counting standard nucleotide characters A, C, G, or T (case-insensitive)
537 strains passed all filters



In [9]:
print("3 - Running augur align")

!~/miniconda3/bin/conda run -n nextstrain augur align \
        --sequences "{RESULTS}/filtered_sequences.fasta" \
        --reference-sequence "{REFSEQ}" \
        --output "{RESULTS}/aligment.fasta" \
        --fill-gaps \
        --remove-reference \
        --nthreads "{TREADS}"


3 - Running augur align

using mafft to align via:
	mafft --reorder --anysymbol --nomemsave --adjustdirection --thread 4 /Users/dt14/Desktop/HCoV/REFs/229E/augur_results/aligment.fasta.to_align.fasta 1> /Users/dt14/Desktop/HCoV/REFs/229E/augur_results/aligment.fasta 2> /Users/dt14/Desktop/HCoV/REFs/229E/augur_results/aligment.fasta.log 

	Katoh et al, Nucleic Acid Research, vol 30, issue 14
	https://doi.org/10.1093%2Fnar%2Fgkf436

Trimmed gaps in NC_002645.1 from the alignment



In [10]:
print("4 - Running augur tree")

#this command can also be run by cmaple
#but the iq-tree3 now has cmapes into it
#cmaple -aln $RESULTS/aligment.fasta --sprta -nt $TREADS

!~/miniconda3/bin/conda run -n nextstrain augur tree \
        --alignment "{RESULTS}/aligment.fasta" \
        --nthreads "{TREADS}" \
        --output "{RESULTS}/aligment.fasta.treefile"

4 - Running augur tree
Building a tree via:
	iqtree3 --threads-max 4 -s /Users/dt14/Desktop/HCoV/REFs/229E/augur_results/aligment-delim.fasta -m GTR --ninit 2 -n 2 --epsilon 0.05 -T AUTO --redo > /Users/dt14/Desktop/HCoV/REFs/229E/augur_results/aligment-delim.iqtree.log
	Please use the corresponding citations according to
	https://iqtree.github.io/doc/Home#how-to-cite-iq-tree

Building original tree took 25.859884023666382 seconds



In [16]:
print("5 - Running augur refine")

!~/miniconda3/bin/conda run -n nextstrain augur refine \
        --tree "{RESULTS}/aligment.fasta.treefile" \
        --alignment "{RESULTS}/aligment.fasta" \
        --metadata "{RESULTS}/filtered_metadata.tsv" \
        --output-tree "{RESULTS}/time_solved.tree" \
        --output-node-data "{RESULTS}/time_solved_node_data.json" \
        --timetree \
        --stochastic-resolve \
        --coalescent 'opt' \
        --date-confidence \
        --date-inference 'joint' \
        --root 'best' \
        --seed 1234567



5 - Running augur refine
augur refine is using TreeTime version 0.11.4

8.74	TreeTime.reroot: with method or node: least-squares

8.74	TreeTime.reroot: rerooting will account for covariance and shared ancestry.

12.70	###TreeTime.run: INITIAL ROUND

21.25	TreeTime.reroot: with method or node: least-squares

21.25	TreeTime.reroot: rerooting will account for covariance and shared
     	ancestry.

21.48	###TreeTime.run: rerunning timetree after rerooting

31.77	###TreeTime.run: ITERATION 1 out of 2 iterations

46.39	###TreeTime.run: ITERATION 2 out of 2 iterations

96.12	###TreeTime.run: FINAL ROUND - confidence estimation via marginal
     	reconstruction

Inferred a time resolved phylogeny using TreeTime:
	Sagulenko et al. TreeTime: Maximum-likelihood phylodynamic analysis
	Virus Evolution, vol 4, https://academic.oup.com/ve/article/4/1/vex042/4794731

updated tree written to /Users/dt14/Desktop/HCoV/REFs/229E/augur_results/time_solved.tree
node attributes written to /Users/dt14/Desktop

In [26]:
#running all these steps at once since they are not very time consuming

# 6 - Running augur ancestral
print("6 - Running augur ancestral")

!~/miniconda3/bin/conda run --no-capture-output -n nextstrain augur ancestral \
    --tree "{RESULTS}/time_solved.tree" \
    --alignment "{RESULTS}/aligment.fasta" \
    --root-sequence "{REFSEQ}" \
    --output-node-data "{RESULTS}/ancestral.json"


# 7 - Running augur traits
print("7 - Running augur traits")

!~/miniconda3/bin/conda run --no-capture-output -n nextstrain augur traits \
    --tree "{RESULTS}/time_solved.tree" \
    --metadata "{RESULTS}/filtered_metadata.tsv" \
    --output-node-data "{RESULTS}/traits.json" \
    --columns Country genotype  \
    --branch-labels Country genotype  \
    --confidence \
    --branch-confidence Country=70 genotype=10


# 8 - Running augur translate
print("8 - Running augur translate")

!~/miniconda3/bin/conda run --no-capture-output -n nextstrain augur translate \
    --tree "{RESULTS}/time_solved.tree" \
    --ancestral-sequences "{RESULTS}/ancestral.json" \
    --reference-sequence "{REFSEQ}" \
    --output-node-data "{RESULTS}/translate.json" \
    --alignment-output "{RESULTS}/my_alignment_%GENE.fasta"


# 9 - Running augur clades (optional)
# print("9 - Running augur clades")
#
# !~/miniconda3/bin/conda run --no-capture-output -n nextstrain augur clades \
#     --tree "{RESULTS}/time_solved.tree" \
#     --mutations "{RESULTS}/traits.json" \
#     --clades "{CLADES}" \
#     --output-node-data "{RESULTS}/clades.json"


# 10 - Running augur frequencies
print("10 - Running augur frequencies")

!~/miniconda3/bin/conda run --no-capture-output -n nextstrain augur frequencies \
    --metadata "{RESULTS}/filtered_metadata.tsv" \
    --tree "{RESULTS}/time_solved.tree" \
    --method kde \
    --pivot-interval 1 \
    --include-internal-nodes \
    --pivot-interval-units weeks \
    --output "{RESULTS}/auspice_output_tip-frequencies.json"



6 - Running augur ancestral

Inferred ancestral sequence states using TreeTime:
	Sagulenko et al. TreeTime: Maximum-likelihood phylodynamic analysis
	Virus Evolution, vol 4, https://academic.oup.com/ve/article/4/1/vex042/4794731

augur ancestral is using TreeTime version 0.11.4
Validating schema of '/Users/dt14/Desktop/HCoV/REFs/229E/augur_results/ancestral.json'...
ancestral mutations written to /Users/dt14/Desktop/HCoV/REFs/229E/augur_results/ancestral.json
7 - Running augur traits
augur traits is using TreeTime version 0.11.4
Assigned discrete traits to 537 out of 537 taxa.

Assigned discrete traits to 537 out of 537 taxa.


Inferred ancestral states of discrete character using TreeTime:
	Sagulenko et al. TreeTime: Maximum-likelihood phylodynamic analysis
	Virus Evolution, vol 4, https://academic.oup.com/ve/article/4/1/vex042/4794731

results written to /Users/dt14/Desktop/HCoV/REFs/229E/augur_results/traits.json
8 - Running augur translate
Read in 8 features from reference sequence

In [27]:
# 11 - Running augur export
print("11 - Running augur export")

!~/miniconda3/bin/conda run --no-capture-output -n nextstrain augur export v2 \
    --tree "{RESULTS}/time_solved.tree" \
    --metadata "{RESULTS}/filtered_metadata.tsv" \
    --node-data "{RESULTS}/time_solved_node_data.json" "{RESULTS}/traits.json" "{RESULTS}/ancestral.json" "{RESULTS}/translate.json" \
    --auspice-config "{CONFIG}" \
    --description "" \
    --output "{RESULTS}/auspice_output.json"
    #--colors "{COLORS}" \

11 - Running augur export
Validating schema of '/Users/dt14/Desktop/HCoV/REFs/229E/augur_results/ancestral.json'...
Validating schema of '/Users/dt14/Desktop/HCoV/REFs/229E/augur_results/translate.json'...
Validating config file '/Users/dt14/Desktop/HCoV/REFs/229E/NCBI_virus/229E_auspice_config.json' against the JSON schema
Validating schema of '/Users/dt14/Desktop/HCoV/REFs/229E/NCBI_virus/229E_auspice_config.json'...




Validating produced JSON
Validating schema of '/Users/dt14/Desktop/HCoV/REFs/229E/augur_results/auspice_output.json'...
Validating that the JSON is internally consistent...
		This will cause transmissions & demes involving this location not to be displayed in Auspice
		This will cause transmissions & demes involving this location not to be displayed in Auspice
Validation of '/Users/dt14/Desktop/HCoV/REFs/229E/augur_results/auspice_output.json' succeeded, but there were warnings you may want to resolve.

